<a href="https://colab.research.google.com/github/evgeniy-borisov/vairl/blob/main/notebooks/pytorch-to-browser-onnx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch → ONNX → браузер (ONNX Runtime Web)

Обучаем лёгкую CNN на MNIST, экспортируем в **ONNX** с именами `input` / `logits` и проверяем инференс — тот же формат используется в [интерактиве VAIRL](https://evgeniy-borisov.github.io/vairl/blog/2026/07/01/pytorch-edge-browser-onnx-ru/).

**VAIRL** — Virtual AI Research Lab

## 1. Зависимости

In [ ]:
!pip install -q torch torchvision onnx onnxruntime matplotlib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2. Модель TinyMNISTCNN

Два свёрточных блока + классификатор. Параметров ~50K — удобно для WASM в браузере.

In [ ]:
class TinyMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(16 * 7 * 7, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = TinyMNISTCNN().to(device)
print(model)

## 3. Обучение на MNIST

Нормализация совпадает с препроцессингом в браузере: mean `0.1307`, std `0.3081`.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_ds = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_ds = datasets.MNIST('./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    print(f'Epoch {epoch+1}/{EPOCHS}  train loss: {total_loss/len(train_ds):.4f}')

model.eval()
correct, total = 0, 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb).argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += yb.size(0)
print(f'Test accuracy: {100 * correct / total:.2f}%')

## 4. Экспорт в ONNX

Имена **`input`** и **`logits`** используются в `edge-nn-webcam-demo.js`. Форма: `[batch, 1, 28, 28]`.

In [ ]:
model.eval()
dummy = torch.randn(1, 1, 28, 28, device=device)
onnx_path = 'mnist-digit-cnn.onnx'

torch.onnx.export(
    model,
    dummy,
    onnx_path,
    input_names=['input'],
    output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)

import os
print('Saved:', onnx_path, f'({os.path.getsize(onnx_path)} bytes)')
if os.path.exists(onnx_path + '.data'):
    print('External weights:', onnx_path + '.data')

## 5. Проверка ONNX (onnxruntime)

In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
inp_name = sess.get_inputs()[0].name
out_name = sess.get_outputs()[0].name
print('ONNX I/O:', inp_name, '->', out_name)

xb, yb = next(iter(test_loader))
xb_np = xb[:1].numpy()
ort_out = sess.run([out_name], {inp_name: xb_np})[0]
pt_out = model(xb[:1].to(device)).detach().cpu().numpy()
print('Max diff PyTorch vs ORT:', np.abs(ort_out - pt_out).max())
print('ORT prediction:', ort_out.argmax(), '| label:', yb[0].item())

## 6. Визуализация примеров

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, (img, label) in zip(axes.ravel(), test_ds[:10]):
    with torch.no_grad():
        logits = model(img.unsqueeze(0).to(device))
        pred = logits.argmax().item()
    denorm = img.squeeze().numpy() * 0.3081 + 0.1307
    ax.imshow(denorm, cmap='gray')
    ax.set_title(f'{pred} ({label})')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Дальше: браузер

1. Скопируйте `mnist-digit-cnn.onnx` (и `.onnx.data`, если есть) в `assets/models/` репозитория VAIRL.
2. Загрузите через ONNX Runtime Web:

```javascript
const session = await ort.InferenceSession.create(modelUrl, {
  executionProviders: ['wasm'],
});
const out = await session.run({ input: tensor });
```

3. Препроцессинг с камеры: grayscale 28×28, `(gray - 0.1307) / 0.3081`.

Готовый виджет: [статья VAIRL](https://evgeniy-borisov.github.io/vairl/blog/2026/07/01/pytorch-edge-browser-onnx-ru/).

In [ ]:
try:
    from google.colab import files
    files.download('mnist-digit-cnn.onnx')
    if os.path.exists('mnist-digit-cnn.onnx.data'):
        files.download('mnist-digit-cnn.onnx.data')
except ImportError:
    print('Локальный запуск: файлы в текущей директории')